# Test Embedder V2 vs V1

Compare performance of the improved embedder (v2) against the original (v1).

- **v1**: Original CLIP-style, width=16, rot=15°, elastic=0.06
- **v2**: Triplet loss, width=24, rot=25°, elastic=0.10, flips, stronger aug

Expected: v2 macro ≈ 0.59+ (vs v1 = 0.557)

In [ ]:
import os
import sys
from pathlib import Path

# Setup paths
DATA_ROOT = Path("/workspace/data/ehl")
WORK_DIR = Path("/workspace/out")
SHARED = Path("/shared-docker/mohamed")

# Add project to path
sys.path.insert(0, str(SHARED))
sys.path.insert(0, "/shared-docker/amine")

# Import eval harness
from eval_harness import (
    build_image_index, cached_volume, read_csv, score_mrr,
    simulate_d2, simulate_d3, DATA_ROOT, GRID, N_VAL, SEED
)
from learned_embedder import build as build_v1

# Import v2
sys.path.insert(0, str(SHARED))
import importlib.util
spec = importlib.util.spec_from_file_location("learned_embedder_v2", 
                                                SHARED / "learned_embedder_v2.py")
learned_v2 = importlib.util.module_from_spec(spec)
spec.loader.exec_module(learned_v2)
build_v2 = learned_v2.build

print(f"Data: {DATA_ROOT}")
print(f"Output: {WORK_DIR}")
print(f"Grid: {GRID}, N_val: {N_VAL}, Seed: {SEED}")

In [ ]:
# Build index and load pairs
import numpy as np

print("Building image index...")
index = build_image_index(DATA_ROOT)
print(f"Found {len(index)} images")

# Load dataset1 pairs
pairs = read_csv(DATA_ROOT / "dataset1" / "train_pairs.csv")
print(f"Total pairs: {len(pairs)}")

# Split into train/val
np.random.seed(SEED)
indices = np.arange(len(pairs))
np.random.shuffle(indices)
val_idx = set(indices[:N_VAL])
train_pairs = [p for i, p in enumerate(pairs) if i not in val_idx]
val_pairs = [p for i, p in enumerate(pairs) if i in val_idx]

print(f"Train: {len(train_pairs)}, Val: {len(val_pairs)}")

In [ ]:
# Train v1 (original)
print("\n" + "="*60)
print("Training V1 (Original)")
print("="*60)

embed_v1 = build_v1(train_pairs, index, GRID, cached_volume)
print(f"V1 trained. Embedding dim: {len(embed_v1(cached_volume(val_pairs[0]['query_id'], index[val_pairs[0]['query_id']], GRID)))}")

In [ ]:
# Train v2 (improved)
print("\n" + "="*60)
print("Training V2 (Improved)")
print("="*60)

embed_v2 = build_v2(train_pairs, index, GRID, cached_volume)
print(f"V2 trained. Embedding dim: {len(embed_v2(cached_volume(val_pairs[0]['query_id'], index[val_pairs[0]['query_id']], GRID)))}")

In [ ]:
# Evaluate on d1/d2/d3 proxies
import time

def eval_all(embed_fn, name):
    """Evaluate on d1, d2, d3 proxies."""
    print(f"\nEvaluating {name}...")
    results = {}
    
    # d1: unmodified
    t0 = time.time()
    d1_score = score_mrr(val_pairs, index, GRID, cached_volume, embed_fn)
    results['d1'] = d1_score
    print(f"  d1 (modality): {d1_score:.4f} ({time.time()-t0:.1f}s)")
    
    # d2: + deformation
    t0 = time.time()
    d2_score = score_mrr(val_pairs, index, GRID, cached_volume, embed_fn, augment_fn=simulate_d2)
    results['d2'] = d2_score
    print(f"  d2 (deform):   {d2_score:.4f} ({time.time()-t0:.1f}s)")
    
    # d3: + structural change
    t0 = time.time()
    d3_score = score_mrr(val_pairs, index, GRID, cached_volume, embed_fn, augment_fn=simulate_d3)
    results['d3'] = d3_score
    print(f"  d3 (struct):   {d3_score:.4f} ({time.time()-t0:.1f}s)")
    
    macro = (results['d1'] + results['d2'] + results['d3']) / 3
    results['macro'] = macro
    print(f"  MACRO:         {macro:.4f}")
    
    return results

# Note: score_mrr needs to be imported from eval_harness
# If it's not available, we can compute it manually

In [ ]:
# Check what's in eval_harness
import eval_harness
print("Available in eval_harness:")
print([x for x in dir(eval_harness) if not x.startswith('_')])

In [ ]:
# Manual MRR scoring
def compute_mrr(val_pairs, index, grid, load_fn, embed_fn, augment_fn=None):
    """Compute Mean Reciprocal Rank."""
    mrrs = []
    
    for pair in val_pairs:
        q_id = pair['query_id']
        t_id = pair['target_id']
        
        q_vol = load_fn(q_id, index[q_id], grid)
        t_vol = load_fn(t_id, index[t_id], grid)
        
        if augment_fn:
            q_vol = augment_fn(q_vol)
            t_vol = augment_fn(t_vol)
        
        z_q = embed_fn(q_vol)
        z_t = embed_fn(t_vol)
        
        # Compute similarity to all targets in val set
        sims = [np.dot(z_q, embed_fn(load_fn(p['target_id'], index[p['target_id']], grid))) 
                for p in val_pairs]
        
        rank = 1 + np.argsort(sims)[::-1].tolist().index(val_pairs.index(pair))
        mrrs.append(1.0 / rank)
    
    return np.mean(mrrs)

print("Ready to compute MRR. (Simple implementation)")

In [ ]:
# Simpler: just compute on a subset for speed
print("Computing scores on validation set...")
print(f"\nV1 Results:")

# d1
z_q1 = np.array([embed_v1(cached_volume(p['query_id'], index[p['query_id']], GRID)) for p in val_pairs])
z_t1 = np.array([embed_v1(cached_volume(p['target_id'], index[p['target_id']], GRID)) for p in val_pairs])
sim1 = np.sum(z_q1 * z_t1, axis=1)
d1_v1 = np.mean([1.0 / (1 + np.sum(sim1 > sim1[i])) for i in range(len(val_pairs))])
print(f"  d1: {d1_v1:.4f}")

print(f"\nV2 Results:")
z_q2 = np.array([embed_v2(cached_volume(p['query_id'], index[p['query_id']], GRID)) for p in val_pairs])
z_t2 = np.array([embed_v2(cached_volume(p['target_id'], index[p['target_id']], GRID)) for p in val_pairs])
sim2 = np.sum(z_q2 * z_t2, axis=1)
d1_v2 = np.mean([1.0 / (1 + np.sum(sim2 > sim2[i])) for i in range(len(val_pairs))])
print(f"  d1: {d1_v2:.4f}")

print(f"\nImprovement: {((d1_v2 - d1_v1) / d1_v1 * 100):+.1f}%")

## Next Steps

If V2 shows improvement:
1. Run full eval_harness.py with v2 on d1/d2/d3
2. If macro > 0.59: proceed to Tier 2 (TTA + loss refinement)
3. Submit best version to Kaggle

If V2 is slower or worse:
1. Analyze bottlenecks
2. Try reducing batch size or network width
3. Or skip v2 and try different improvements (e.g., ensemble)